# Ei, Emily — novo banco de vozes (piloto)

**Teste de uma nova fonte de áudio; não instala nada nem treina outro detector ainda.**

1. No Colab, selecione GPU **T4** e execute tudo.
2. Baixe o ZIP ao terminar e devolva para avaliação.

Gera 12 perfis solicitados em português brasileiro, com descrições de seis regiões, adultos de idades diferentes e timbres variados. Qwen3-TTS VoiceDesign cria referências sintéticas; Qwen3-TTS Base 0.6B reutiliza cada referência para duas chamadas e dois negativos. Não usa sua gravação, nem clona pessoas reais. **São perfis sintéticos: sotaques e diferenças entre vozes precisam ser confirmados ouvindo; não são 12 falantes reais comprovados.**

Whisper auxilia a transcrição; alertas de duração, prefixo ausente, nomes isolados e embeddings muito semelhantes ficam no relatório. ASR e embeddings não comprovam pronúncia correta ou diversidade real. A amostra é deliberadamente pequena antes de gastar horas com até 200 perfis.

O ambiente fica em `/content/emily-voice-source-pilot`, separado do treinamento anterior. Cada modelo é carregado em um processo independente. Na T4 usamos FP32 para evitar problemas de FP16; a síntese pode demorar. O tempo real fica registrado no ZIP, permitindo estimar se centenas de perfis cabem no Colab gratuito. Não ativa cobrança nem conecta Google Drive. Se o Colab interromper, execute novamente para aproveitar os WAVs concluídos, enquanto a máquina temporária existir. Não é necessário reexecutar o treinamento v2/v3.

As versões dos modelos estão fixadas e sua licença é Apache-2.0. O resultado contém apenas áudios sintéticos e seus registros. Ainda faltam sons/conversas reais para treinar e avaliar falsos acionamentos.


In [ ]:
from pathlib import Path
import json, os, subprocess
root = Path('/content/emily-voice-source-pilot')
root.mkdir(exist_ok=True)
sources = json.loads("{\"voice_bank_pilot.py\": \"\\\"\\\"\\\"Audit a new synthetic PT-BR voice bank before any wake-word training.\\\"\\\"\\\"\\nfrom pathlib import Path\\nimport argparse\\nimport hashlib\\nimport itertools\\nimport json\\nimport math\\nimport time\\nimport unicodedata\\nimport numpy as np\\nfrom scipy.signal import resample_poly\\n\\nMODELS = {\\n    'reference': ('Qwen/Qwen3-TTS-12Hz-1.7B-VoiceDesign', '5ecdb67327fd37bb2e042aab12ff7391903235d3'),\\n    'phrases': ('Qwen/Qwen3-TTS-12Hz-0.6B-Base', '5d83992436eae1d760afd27aff78a71d676296fc'),\\n}\\nREFERENCE_TEXT = 'Hoje eu vou caminhar pelo bairro e conversar com meus amigos. Depois volto para casa e escuto uma música.'\\nPHRASES = [('call-emily', True, 'Ei, Emily.'), ('call-emili', True, 'Ei, Emili.'),\\n           ('name-only', False, 'Emily.'), ('mention', False, 'Eu estava falando com a Emily ontem.')]\\nREGIONS = ['São Paulo', 'Rio de Janeiro', 'Minas Gerais', 'Bahia', 'Pernambuco', 'Rio Grande do Sul']\\nAGES = ['young adult', 'middle-aged adult', 'older adult']\\nTIMBRES = ['clear and bright', 'warm and resonant', 'soft and airy', 'slightly husky', 'low and rounded', 'light and nasal']\\n\\ndef digest(path):\\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\\n\\ndef profiles(count):\\n    if not 1 <= count <= 216:\\n        raise ValueError('Use de 1 a 216 perfis solicitados; o piloto recomendado usa 12.')\\n    result = []\\n    for cycle in range(18):\\n        for region_index, region in enumerate(REGIONS):\\n            for gender_index, gender in enumerate(['female', 'male']):\\n                age = AGES[(cycle+region_index+gender_index)%len(AGES)]\\n                timbre = TIMBRES[(cycle//3+region_index+gender_index)%len(TIMBRES)]\\n                instruction = (f'Natural Brazilian Portuguese, a {age} {gender} voice with a {timbre} timbre. '\\n                               f'A gentle everyday regional accent from {region}, Brazil. '\\n                               'Speak conversationally, without caricature, singing or theatrical exaggeration.')\\n                ident = hashlib.sha256(instruction.encode()).hexdigest()[:16]\\n                if any(p['id']==ident for p in result):\\n                    continue\\n                result.append(dict(id=ident,requested_region=region,requested_age=age,\\n                                   requested_gender=gender,requested_timbre=timbre,instruction=instruction))\\n    if len(result)<count:\\n        raise RuntimeError('Descrições insuficientes; não inflar o banco por duplicação.')\\n    return result[:count]\\n\\ndef normalize(text):\\n    text = ''.join(c for c in unicodedata.normalize('NFKD',text.casefold()) if not unicodedata.combining(c))\\n    return ' '.join(''.join(c if c.isalpha() else ' ' for c in text).split())\\n\\ndef transcript_flags(text, positive):\\n    words = normalize(text).split()\\n    names = {'emily','emili','emilie'}\\n    call = len(words)>=2 and words[0] in {'ei','hey'} and words[1] in names\\n    if positive:\\n        return ([] if call and len(words)==2 else ['call_transcription_needs_listening'])\\n    return ['negative_transcription_contains_call'] if any(\\n        words[i] in {'ei','hey'} and words[i+1] in names for i in range(len(words)-1)) else []\\n\\ndef audio_checks(audio,rate,positive=False):\\n    audio = np.asarray(audio,dtype=np.float32)\\n    if rate<8000 or audio.ndim!=1 or len(audio)<rate*.1 or not np.isfinite(audio).all():\\n        raise ValueError('Áudio vazio, inválido ou não mono.')\\n    peak = float(np.abs(audio).max())\\n    if peak<.001:\\n        raise ValueError('Síntese silenciosa.')\\n    frame = max(1,int(rate*.01))\\n    energy = np.sqrt(np.mean(audio[:len(audio)//frame*frame].reshape(-1,frame)**2,axis=1))\\n    active = np.flatnonzero(energy>max(.0005,float(energy.max())*.02))\\n    if not len(active):\\n        raise ValueError('Sem fala detectável.')\\n    active_seconds = float((active[-1]-active[0]+1)*frame/rate)\\n    leading_seconds = float(active[0]*frame/rate)\\n    trailing_seconds = float((len(audio)-(active[-1]+1)*frame)/rate)\\n    flags = []\\n    if positive and active_seconds>2.:\\n        flags.append('call_longer_than_v2_training_budget')\\n    if float(np.mean(np.abs(audio)>=.999))>.01:\\n        flags.append('clipping')\\n    if len(audio)/rate>16:\\n        flags.append('possible_generation_loop_or_truncation')\\n    # A short call can hit the token budget while producing silence rather than speech.\\n    # Keep the raw waveform: an energy boundary is not a safe keyword trimming rule.\\n    if positive and len(audio)/rate>4:\\n        flags.append('short_call_unexpected_file_duration')\\n    if leading_seconds>1:\\n        flags.append('excessive_leading_silence')\\n    if trailing_seconds>1:\\n        flags.append('excessive_trailing_silence')\\n    return dict(seconds=len(audio)/rate,active_seconds=active_seconds,\\n                leading_seconds=leading_seconds,trailing_seconds=trailing_seconds,peak=peak,flags=flags)\\n\\ndef pcm16(audio,rate):\\n    factor = math.gcd(rate,16000)\\n    resampled = resample_poly(np.asarray(audio,np.float32),16000//factor,rate//factor)\\n    if not np.isfinite(resampled).all():\\n        raise ValueError('Resampling inválido.')\\n    return np.clip(np.rint(resampled*32768),-32768,32767).astype(np.int16)\\n\\ndef write_json(path,data):\\n    temporary = path.with_suffix('.part')\\n    temporary.write_text(json.dumps(data,ensure_ascii=False,indent=2))\\n    temporary.replace(path)\\n\\ndef read_progress(work,count):\\n    work.mkdir(parents=True,exist_ok=True)\\n    settings = dict(models=MODELS,profiles=profiles(count),phrases=PHRASES,reference_text=REFERENCE_TEXT,\\n                    script_sha256=digest(__file__),evidence='synthetic_profiles_not_real_speakers')\\n    signature = hashlib.sha256(json.dumps(settings,sort_keys=True,ensure_ascii=False).encode()).hexdigest()\\n    path = work/'progress.json'\\n    data = json.loads(path.read_text()) if path.exists() else dict(signature=signature,settings=settings,records=[])\\n    if data['signature']!=signature:\\n        raise ValueError('Piloto alterado: use uma pasta nova; não misturar vozes ou parâmetros.')\\n    for relative, expected in data.get('embedding_hashes',{}).items():\\n        path = Path(relative)\\n        if path.is_absolute() or '..' in path.parts or digest(work/path)!=expected:\\n            raise ValueError('Embedding do piloto alterado ou caminho inválido.')\\n    for row in data['records']:\\n        relative = Path(row['path'])\\n        if relative.is_absolute() or '..' in relative.parts or digest(work/relative)!=row['sha256']:\\n            raise ValueError('Arquivo do piloto alterado ou caminho inválido.')\\n    return data\\n\\ndef snapshot(stage,work):\\n    from huggingface_hub import snapshot_download\\n    repo,revision = MODELS[stage]\\n    return snapshot_download(repo_id=repo,revision=revision,cache_dir=str(work.parent/'model-cache'),\\n                             allow_patterns=['*.json','*.safetensors','*.txt','*.model','README.md','LICENSE'])\\n\\ndef load_model(stage,work):\\n    import torch\\n    from qwen_tts import Qwen3TTSModel\\n    if not torch.cuda.is_available():\\n        raise RuntimeError('Este piloto precisa da GPU gratuita do Colab; selecione T4.')\\n    # Avoid FP16 sampling overflow on Turing/T4; native BF16 is used only when supported.\\n    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32\\n    print('GPU:',torch.cuda.get_device_name(0),'precision:',dtype,flush=True)\\n    model = Qwen3TTSModel.from_pretrained(snapshot(stage,work),device_map='cuda:0',dtype=dtype,\\n                                        attn_implementation='sdpa',low_cpu_mem_usage=True)\\n    if 'portuguese' not in model.get_supported_languages():\\n        raise RuntimeError('Português não disponível nesta revisão.')\\n    return model\\n\\ndef seed(profile,phrase):\\n    return int(hashlib.sha256((profile+':'+phrase).encode()).hexdigest()[:8],16)\\n\\ndef generate(stage,work,count):\\n    import torch\\n    import soundfile as sf\\n    from scipy.io import wavfile\\n    from transformers import set_seed\\n    data = read_progress(work,count)\\n    if stage=='reference':\\n        missing = [p for p in profiles(count) if not any(r['voice']==p['id'] and r['kind']=='reference' for r in data['records'])]\\n    else:\\n        missing = [p for p in profiles(count) if any(not any(r['voice']==p['id'] and r['kind']==key for r in data['records']) for key,_,_ in PHRASES)]\\n    if not missing:\\n        print('Etapa já concluída:',stage,flush=True)\\n        return\\n    model = load_model(stage,work)\\n    for profile in missing:\\n        ident = profile['id']\\n        print('Perfil solicitado:',profile['requested_region'],profile['requested_gender'],ident,flush=True)\\n        if stage=='reference':\\n            set_seed(seed(ident,'reference'))\\n            started = time.monotonic()\\n            wavs,rate = model.generate_voice_design(text=REFERENCE_TEXT,language='Portuguese',\\n                instruct=profile['instruction'],max_new_tokens=384,temperature=.8)\\n            audio = np.asarray(wavs[0]).reshape(-1)\\n            checks = audio_checks(audio,rate)\\n            if checks['seconds']<3 or checks['seconds']>20:\\n                raise ValueError('Referência com duração inadequada; revisar antes de clonar.')\\n            relative = Path('references')/(ident+'.wav')\\n            (work/relative).parent.mkdir(exist_ok=True)\\n            sf.write(work/relative,audio,rate,subtype='PCM_16')\\n            data['records'].append(dict(voice=ident,kind='reference',path=str(relative),sha256=digest(work/relative),\\n                expected_text=REFERENCE_TEXT,positive=False,model_revision=MODELS[stage][1],seed=seed(ident,'reference'),\\n                inference_seconds=time.monotonic()-started,checks=checks))\\n            write_json(work/'progress.json',data)\\n        else:\\n            reference = next((r for r in data['records'] if r['voice']==ident and r['kind']=='reference'),None)\\n            if reference is None:\\n                raise ValueError('Gere a referência antes das chamadas.')\\n            prompt = model.create_voice_clone_prompt(ref_audio=str(work/reference['path']),ref_text=REFERENCE_TEXT)\\n            vector = prompt[0].ref_spk_embedding.detach().float().cpu().numpy().ravel()\\n            if not np.isfinite(vector).all() or np.linalg.norm(vector)==0:\\n                raise ValueError('Embedding de referência inválido.')\\n            vector_path = Path('references')/(ident+'.npy')\\n            np.save(work/vector_path,vector,allow_pickle=False)\\n            data.setdefault('embedding_hashes',{})[str(vector_path)] = digest(work/vector_path)\\n            write_json(work/'progress.json',data)\\n            for key,positive,text in PHRASES:\\n                if any(r['voice']==ident and r['kind']==key for r in data['records']):\\n                    continue\\n                set_seed(seed(ident,key))\\n                started = time.monotonic()\\n                wavs,rate = model.generate_voice_clone(text=text,language='Portuguese',voice_clone_prompt=prompt,\\n                                                     max_new_tokens=192,temperature=.8)\\n                audio = np.asarray(wavs[0]).reshape(-1)\\n                checks = audio_checks(audio,rate,positive)\\n                relative = Path('audio')/(ident+'-'+key+'.wav')\\n                (work/relative).parent.mkdir(exist_ok=True)\\n                wavfile.write(work/relative,16000,pcm16(audio,rate))\\n                data['records'].append(dict(voice=ident,kind=key,path=str(relative),sha256=digest(work/relative),\\n                    expected_text=text,positive=positive,model_revision=MODELS[stage][1],seed=seed(ident,key),\\n                    inference_seconds=time.monotonic()-started,checks=checks))\\n                write_json(work/'progress.json',data)\\n    print('Etapa concluída:',stage,flush=True)\\n\\n\\ndef audit(work,count):\\n    import whisper\\n    import torch\\n    from difflib import SequenceMatcher\\n    data = read_progress(work,count)\\n    if len(data['records'])!=count*(len(PHRASES)+1):\\n        raise ValueError('Piloto incompleto; conclua referências e frases antes de auditar.')\\n    if not torch.cuda.is_available():\\n        raise RuntimeError('GPU não disponível para a auditoria.')\\n    model = whisper.load_model('small',device='cuda',download_root=str(work.parent/'asr-cache'))\\n    rows = []\\n    for row in data['records']:\\n        result = model.transcribe(str(work/row['path']),language='pt',task='transcribe',fp16=True,\\n                                  temperature=0,condition_on_previous_text=False)\\n        text = result['text'].strip()\\n        flags = list(row['checks']['flags'])\\n        if row['kind']=='reference':\\n            similarity = SequenceMatcher(None,normalize(row['expected_text']),normalize(text)).ratio()\\n            if similarity<.75:\\n                flags.append('reference_transcription_needs_listening')\\n        else:\\n            flags.extend(transcript_flags(text,row['positive']))\\n        rows.append(dict(**row,transcript=text,audit_flags=flags,pronunciation_verified=False))\\n        print(row['kind'],row['voice'],text,flags,flush=True)\\n    vectors = [np.load(work/'references'/(p['id']+'.npy'),allow_pickle=False) for p in profiles(count)]\\n    normalized = np.asarray([v/np.linalg.norm(v) for v in vectors])\\n    similarities = normalized@normalized.T\\n    similar = [dict(first=profiles(count)[a]['id'],second=profiles(count)[b]['id'],cosine=float(similarities[a,b]))\\n               for a,b in itertools.combinations(range(count),2) if similarities[a,b]>.95]\\n    report = dict(status='voice_source_pilot_only_not_a_wakeword_model',synthetic_profiles_requested=count,\\n        real_speakers_recorded=0,accent_verified=False,profiles=data['settings']['profiles'],\\n        similar_embedding_pairs=similar,similarity_note='Heuristic warning only; not a proof of unique speakers or independent accents.',\\n        manual_review_required=True,records=rows,source_models=MODELS,source_license='Apache-2.0',\\n        generation_seconds=sum(r['inference_seconds'] for r in rows),\\n        missing=['human pronunciation and accent audit','held-out speaker grouping before training',\\n                 'real ambient negatives and long-audio evaluation','proof of improvement against Android baseline'])\\n    write_json(work/'audit.json',report)\\n    import html\\n    cards=[]\\n    for profile in profiles(count):\\n        ident=profile['id'];items=[]\\n        for row in rows:\\n            if row['voice']!=ident:continue\\n            items.append('<p>'+html.escape(row['expected_text'])+'</p><audio controls preload=\\\"none\\\" src=\\\"'+row['path']+'\\\"></audio><p class=\\\"small\\\">Whisper: '+html.escape(row['transcript'])+'<br>'+html.escape(', '.join(row['audit_flags']))+'</p>')\\n        cards.append('<section><h2>'+html.escape(profile['requested_region']+' · '+profile['requested_gender'])+'</h2><p class=\\\"small\\\">Sotaque solicitado, ainda não confirmado. '+html.escape(profile['requested_age']+' / '+profile['requested_timbre'])+'</p>'+''.join(items)+'</section>')\\n    (work/'listening.html').write_text('<!doctype html><html lang=\\\"pt-BR\\\"><meta charset=\\\"utf-8\\\"><meta name=\\\"viewport\\\" content=\\\"width=device-width,initial-scale=1\\\"><title>Emily · auditoria de vozes</title><style>body{background:#151513;color:#eee;font:16px system-ui;max-width:850px;margin:auto;padding:24px}section{background:#242421;padding:20px;margin:20px 0;border-radius:18px}audio{width:100%}h1,h2{color:#f29765}.small{color:#aaa;font-size:14px}</style><h1>Auditoria de vozes · Ei, Emily</h1><p>Piloto sintético. Ouça a chamada inteira, o “Ei”, a pronúncia do nome e se as vozes realmente diferem. A transcrição automática é auxiliar e pode errar. Este pacote não atualiza o Android.</p>'+''.join(cards)+'</html>')\\n    import zipfile\\n    archive=work.with_suffix('.zip')\\n    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:\\n        for path in work.rglob('*'):\\n            if path.is_file() and not path.is_symlink():\\n                z.write(path,str(path.relative_to(work)))\\n    print('Piloto pronto:',archive,flush=True)\\n\\n\\ndef main():\\n    parser=argparse.ArgumentParser(description=__doc__)\\n    parser.add_argument('stage',choices=['reference','phrases','audit'])\\n    parser.add_argument('--work',type=Path,required=True)\\n    parser.add_argument('--profiles',type=int,default=12)\\n    args=parser.parse_args()\\n    profiles(args.profiles)\\n    if args.stage=='audit':audit(args.work.resolve(),args.profiles)\\n    else:generate(args.stage,args.work.resolve(),args.profiles)\\n\\nif __name__=='__main__':main()\\n\", \"test_voice_bank_pilot.py\": \"import os\\nimport sys\\nfrom pathlib import Path\\nimport tempfile\\nimport unittest\\nimport numpy as np\\nfrom scipy.io import wavfile\\nfrom setup_voice_bank import run, generator_environment\\nfrom voice_bank_pilot import profiles,transcript_flags,audio_checks,pcm16,read_progress,digest,write_json\\n\\nclass VoiceBankChecks(unittest.TestCase):\\n    def test_count_is_not_inflated_by_repeated_descriptions(self):\\n        rows=profiles(216)\\n        self.assertEqual(len({r['id'] for r in rows}),216)\\n        self.assertEqual(len({r['instruction'] for r in rows}),216)\\n        self.assertEqual(len({r['requested_region'] for r in profiles(12)}),6)\\n    def test_name_alone_and_missing_prefix_not_approved(self):\\n        for text in ['Emily','E Emily','Ei','Emily Ei','Ei Emily abre o YouTube']:\\n            self.assertTrue(transcript_flags(text,True))\\n        self.assertFalse(transcript_flags('Ei, Êmili!',True))\\n        self.assertTrue(transcript_flags('Eu falei ei Emily ontem',False))\\n    def test_invalid_generation_cannot_enter_bank(self):\\n        for audio in [np.zeros(16000),np.full(16000,np.nan),np.ones((2,16000))]:\\n            with self.assertRaises(ValueError):audio_checks(audio,16000)\\n        audio=np.sin(np.arange(3*16000)*.1)*.5\\n        self.assertIn('call_longer_than_v2_training_budget',audio_checks(audio,16000,True)['flags'])\\n    def test_preserve_low_volume_pcm_and_resample_duration(self):\\n        audio=np.sin(np.arange(24000)*.1)*.01\\n        pcm=pcm16(audio,24000)\\n        self.assertEqual(pcm.dtype,np.int16)\\n        self.assertEqual(len(pcm),16000)\\n        self.assertLess(int(np.abs(pcm).max()),500)\\n        self.assertGreater(int(np.abs(pcm).max()),200)\\n    def test_short_call_with_long_silent_tail_is_flagged_without_trimming(self):\\n        speech=np.sin(np.arange(16000)*.1)*.05\\n        audio=np.concatenate([np.zeros(3200),speech,np.zeros(13*16000)])\\n        original=audio.copy()\\n        checks=audio_checks(audio,16000,True)\\n        self.assertIn('short_call_unexpected_file_duration',checks['flags'])\\n        self.assertIn('excessive_trailing_silence',checks['flags'])\\n        self.assertAlmostEqual(checks['leading_seconds'],.2,places=2)\\n        self.assertAlmostEqual(checks['trailing_seconds'],13,places=2)\\n        np.testing.assert_array_equal(audio,original)\\n    def test_driver_paths_preserved_and_training_libs_removed(self):\\n        original = {'LD_LIBRARY_PATH':'/usr/lib64-nvidia:/content/emily-wakeword/.training-venv/lib/python3.11/site-packages/nvidia/cublas/lib:/usr/local/cuda/lib64/stubs',\\n                    'CUDA_VISIBLE_DEVICES':'0'}\\n        env = generator_environment(original)\\n        self.assertIn('/usr/lib64-nvidia',env['LD_LIBRARY_PATH'].split(':'))\\n        self.assertNotIn('.training-venv/',env['LD_LIBRARY_PATH'])\\n        self.assertNotIn('/stubs',env['LD_LIBRARY_PATH'])\\n        self.assertEqual(env['CUDA_VISIBLE_DEVICES'],'0')\\n        self.assertIn('.training-venv/',original['LD_LIBRARY_PATH'])\\n\\n    def test_setup_failure_reports_real_diagnostics(self):\\n        with tempfile.TemporaryDirectory() as temporary:\\n            root=Path(temporary)\\n            with self.assertRaisesRegex(RuntimeError,'EXPECTED_SETUP_FAILURE'):\\n                run([sys.executable,'-c',\\\"print('EXPECTED_SETUP_FAILURE'); raise SystemExit(7)\\\"],root,os.environ.copy())\\n            self.assertIn('EXPECTED_SETUP_FAILURE',(root/'setup.log').read_text())\\n\\n    def test_resume_rejects_modified_audio_and_changed_bank(self):\\n        with tempfile.TemporaryDirectory() as temporary:\\n            root=Path(temporary)\\n            state=read_progress(root,12)\\n            wavfile.write(root/'a.wav',16000,np.ones(16000,np.int16))\\n            state['records'].append(dict(path='a.wav',sha256=digest(root/'a.wav')))\\n            write_json(root/'progress.json',state)\\n            self.assertEqual(len(read_progress(root,12)['records']),1)\\n            with self.assertRaises(ValueError):read_progress(root,13)\\n            (root/'a.wav').write_bytes(b'changed')\\n            with self.assertRaises(ValueError):read_progress(root,12)\\n\\nif __name__=='__main__':unittest.main()\\n\", \"setup_voice_bank.py\": \"\\\"\\\"\\\"Dedicated generator environment; never install into the wakeword/Emily env.\\\"\\\"\\\"\\nfrom pathlib import Path\\nimport hashlib\\nimport os\\nimport shutil\\nimport subprocess\\nimport sys\\n\\n\\ndef run(command,root,env):\\n    with (root/'setup.log').open('a') as log:\\n        with subprocess.Popen([str(x) for x in command],cwd=root,env=env,stdin=subprocess.DEVNULL,\\n                              stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1) as p:\\n            tail=[]\\n            for line in p.stdout:\\n                print(line,end='',flush=True)\\n                log.write(line);log.flush()\\n                tail=(tail+[line])[-25:]\\n            code=p.wait()\\n        if code:\\n            raise RuntimeError(f'Falha na preparação ({code}). Log: {root / \\\"setup.log\\\"}\\\\n'+''.join(tail))\\n\\n\\ndef generator_environment(source):\\n    env = source.copy()\\n    # Keep the NVIDIA driver paths that Colab injects. Only remove known old training libs/stubs.\\n    paths = []\\n    for path in env.get('LD_LIBRARY_PATH','').split(':'):\\n        if not path or '.training-venv/' in path or path.endswith('/stubs'):\\n            continue\\n        paths.append(path)\\n    for path in ['/usr/lib64-nvidia','/usr/local/nvidia/lib64','/usr/local/nvidia/lib']:\\n        if Path(path).is_dir() and path not in paths:\\n            paths.append(path)\\n    if paths:\\n        env['LD_LIBRARY_PATH'] = ':'.join(paths)\\n    else:\\n        env.pop('LD_LIBRARY_PATH',None)\\n    env['OMP_NUM_THREADS']='2';env['OPENBLAS_NUM_THREADS']='2'\\n    env['HF_HUB_DISABLE_TELEMETRY']='1'\\n    return env\\n\\n\\ndef setup(root):\\n    root=Path(root).resolve();root.mkdir(parents=True,exist_ok=True)\\n    env=generator_environment(os.environ)\\n    for executable in ['ffmpeg','sox']:\\n        if not shutil.which(executable):\\n            if not str(root).startswith('/content/'):\\n                raise RuntimeError('Este preparador é para Colab; falta '+executable)\\n            run(['apt-get','update','-qq'],root,env)\\n            run(['apt-get','install','-y','-qq',executable],root,env)\\n    python=root/'.voicegen-venv/bin/python'\\n    if not python.exists():\\n        old=Path('/content/emily-wakeword/.training-venv/bin/python')\\n        if old.is_file():\\n            run([old,'-m','venv',python.parent.parent],root,env)\\n        else:\\n            bootstrap=root/'.bootstrap'\\n            uv=bootstrap/'bin/uv'\\n            if not uv.is_file():\\n                run([sys.executable,'-m','pip','install','--target',bootstrap,'uv==0.8.22'],root,env)\\n            env['UV_PYTHON_INSTALL_DIR']=str(root/'.managed-python')\\n            env['UV_CACHE_DIR']=str(root/'.uv-cache');env['UV_NATIVE_TLS']='true'\\n            run([uv,'venv','--managed-python','--python','3.11','--seed',python.parent.parent],root,env)\\n    if shutil.disk_usage(root).free<20*1024**3:\\n        raise RuntimeError('Menos de 20 GiB livres no Colab. Não iniciar downloads dos modelos.')\\n    signature=hashlib.sha256((root/'requirements-voice-pilot.txt').read_bytes()+b'torch2.6.0+cu124').hexdigest()\\n    ready=root/'.dependencies-ready'\\n    if ready.exists() and ready.read_text()!=signature:\\n        raise RuntimeError('Dependências alteradas; use outra pasta isolada.')\\n    if not ready.exists():\\n        run([python,'-m','pip','install','torch==2.6.0','torchaudio==2.6.0',\\n             '--index-url','https://download.pytorch.org/whl/cu124'],root,env)\\n        run([python,'-m','pip','install','-r',root/'requirements-voice-pilot.txt'],root,env)\\n        run([python,'-m','pip','check'],root,env)\\n        ready.write_text(signature)\\n    run([python,'-m','unittest','test_voice_bank_pilot'],root,env)\\n    try:\\n        run([python,root/'voice_gpu_probe.py'],root,env)\\n    except RuntimeError as error:\\n        raise RuntimeError('A GPU não inicializou no ambiente do gerador. Isso não prova que a T4 '\\n                           'não foi selecionada. O diagnóstico acima identifica driver, visibilidade '\\n                           'e erro real do PyTorch; preserve esta sessão e seus arquivos.') from error\\n    run([python,'-c','from qwen_tts import Qwen3TTSModel; import whisper; '\\n         'print(\\\"Bibliotecas do gerador carregadas.\\\")'],root,env)\\n    return str(python),env\\n\", \"voice_gpu_probe.py\": \"\\\"\\\"\\\"Small GPU probe for a fresh subprocess; prints only GPU-related diagnostics.\\\"\\\"\\\"\\nimport ctypes\\nimport json\\nimport os\\nfrom pathlib import Path\\nimport subprocess\\nimport sys\\n\\ndef probe():\\n    result = dict(python=sys.version.split()[0],executable=sys.executable,\\n                  environment={k:os.environ.get(k) for k in ['CUDA_VISIBLE_DEVICES','NVIDIA_VISIBLE_DEVICES',\\n                    'LD_LIBRARY_PATH','PYTORCH_NVML_BASED_CUDA_CHECK']})\\n    try:\\n        p = subprocess.run(['nvidia-smi','--query-gpu=name,driver_version,memory.total','--format=csv,noheader'],\\n                           capture_output=True,text=True,timeout=15)\\n        result['nvidia_smi']=dict(returncode=p.returncode,stdout=p.stdout.strip(),stderr=p.stderr.strip())\\n    except (OSError,subprocess.TimeoutExpired) as error:\\n        result['nvidia_smi']=dict(error=str(error))\\n    try:\\n        driver=ctypes.CDLL('libcuda.so.1')\\n        driver.cuInit.argtypes=[ctypes.c_uint];driver.cuInit.restype=ctypes.c_int\\n        code=driver.cuInit(0)\\n        result['driver_init_code']=code\\n        name=ctypes.c_char_p()\\n        driver.cuGetErrorName.argtypes=[ctypes.c_int,ctypes.POINTER(ctypes.c_char_p)]\\n        driver.cuGetErrorName(code,ctypes.byref(name))\\n        result['driver_init_name']=name.value.decode() if name.value else None\\n        count=ctypes.c_int()\\n        result['driver_device_count_code']=driver.cuDeviceGetCount(ctypes.byref(count))\\n        result['driver_device_count']=count.value\\n    except (OSError,AttributeError) as error:\\n        result['driver_error']=str(error)\\n    try:\\n        import torch\\n        result.update(torch_version=torch.__version__,torch_cuda_build=torch.version.cuda,\\n                      cuda_available=torch.cuda.is_available())\\n        try:\\n            torch.cuda.init()\\n            result['torch_gpu_name']=torch.cuda.get_device_name(0)\\n            # Initialization alone doesn't verify the CUDA runtime can execute a kernel.\\n            result['kernel_test']=float((torch.ones(4,device='cuda')*2).sum().item())\\n        except Exception as error:\\n            result['torch_init_error']=f'{type(error).__name__}: {error}'\\n    except ImportError as error:\\n        result['torch_import_error']=str(error)\\n    try:\\n        result['loaded_driver_paths']=sorted({line.split()[-1] for line in Path('/proc/self/maps').read_text().splitlines()\\n                                             if 'libcuda.so' in line and line.split()[-1].startswith('/')})\\n    except OSError:\\n        pass\\n    return result\\n\\nif __name__=='__main__':\\n    result=probe()\\n    encoded=json.dumps(result,ensure_ascii=False,indent=2)\\n    Path(__file__).with_name('gpu-diagnostics.json').write_text(encoded)\\n    print(encoded)\\n    raise SystemExit(0 if result.get('kernel_test')==8. else 1)\\n\", \"requirements-voice-pilot.txt\": \"qwen-tts==0.1.1\\nopenai-whisper==20250625\\nnumpy==1.26.4\\nscipy==1.15.3\\nhuggingface-hub==0.36.0\\nsoundfile==0.13.1\\nonnxruntime==1.23.2\\n\"}")
for name, contents in sources.items(): (root/name).write_text(contents)
os.chdir(root)
import sys
sys.path.insert(0,str(root))
import importlib, setup_voice_bank
importlib.reload(setup_voice_bank)
from setup_voice_bank import setup
python, env = setup(root)


In [ ]:
# Piloto de qualidade: não ampliar antes de ouvir/analisar o resultado.
PROFILES = 12
work = root/'work-source-pilot-12-audit2'
for stage in ['reference','phrases','audit']:
    print('Etapa:',stage,flush=True)
    subprocess.run([python,str(root/'voice_bank_pilot.py'),stage,'--work',str(work),
                    '--profiles',str(PROFILES)],cwd=root,env=env,check=True)
from IPython.display import Audio, display
report = json.loads((work/'audit.json').read_text())
print('Requer avaliação humana; não é um modelo do Android.')
for profile in report['profiles']:
    print(profile['requested_region'],profile['requested_gender'],'— sotaque solicitado, não confirmado')
    example = next(row for row in report['records'] if row['voice']==profile['id'] and row['kind']=='call-emily')
    display(Audio(filename=str(work/example['path']), normalize=False))
from google.colab import files
files.download(str(work.with_suffix('.zip')))
